In [1]:
!pip install networkx

In [2]:
import networkx as nx

# Create graph
G = nx.DiGraph()

# Add relationships manually (simulating LLM extraction)
triples = [
    ("OpenAI", "developed", "GPT-4"),
    ("Microsoft", "invested_in", "OpenAI"),
    ("GPT-4", "powers", "ChatGPT"),
]

for subj, rel, obj in triples:
    G.add_edge(subj, obj, relation=rel)

print("Graph nodes:", G.nodes())
print("Graph edges:", G.edges(data=True))

Graph nodes: ['OpenAI', 'GPT-4', 'Microsoft', 'ChatGPT']
Graph edges: [('OpenAI', 'GPT-4', {'relation': 'developed'}), ('GPT-4', 'ChatGPT', {'relation': 'powers'}), ('Microsoft', 'OpenAI', {'relation': 'invested_in'})]


In [ ]:
## Simple Graph Retrieval
#Find how Microsoft connects to ChatGPT
import networkx as nx

def find_relation_path(graph, source, target):
    try:
        path = nx.shortest_path(graph, source=source, target=target)
        return path
    except nx.NetworkXNoPath:
        return None

path = find_relation_path(G, "Microsoft", "ChatGPT")

print("Path:", path)

Path: ['Microsoft', 'OpenAI', 'GPT-4', 'ChatGPT']


In [4]:
#Convert Path to Context for LLM
def build_context_from_path(graph, path):
    context = []
    for i in range(len(path)-1):
        edge_data = graph.get_edge_data(path[i], path[i+1])
        relation = edge_data["relation"]
        context.append(f"{path[i]} {relation} {path[i+1]}")
    return ". ".join(context)

context = build_context_from_path(G, path)
print("Context for LLM:")
print(context)

Context for LLM:
Microsoft invested_in OpenAI. OpenAI developed GPT-4. GPT-4 powers ChatGPT


In [ ]:
from openai import OpenAI
client = OpenAI(api_key="")

question = "How is Microsoft related to ChatGPT?"

prompt = f"""
Use the following knowledge graph facts:

{context}

Answer the question:
{question}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Microsoft is related to ChatGPT through its investment in OpenAI, the organization that developed GPT-4, which powers ChatGPT.
